# CODI Training on GPU - TokenSkip Thesis

This notebook trains CODI-GPT2 on your GSM8K split for Phase 1 of continuous reasoning steering.

**GPU**: Free T4 (Runtime → Change runtime type → T4 GPU)

**Time**: ~2 hours for 3 epochs

In [ ]:
# Verify GPU
import torch
print(f"✓ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"✓ Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Option A: Clone from GitHub (recommended)
!git clone https://github.com/nabilanewaz/TokenSkip.git
%cd TokenSkip

In [ ]:
# Option B: Or mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/TokenSkip

In [ ]:
# Install dependencies
!pip install peft==0.15.2 datasets==3.6.0 huggingface_hub transformers==4.52.4 accelerate==1.7.0 safetensors -q
print("✓ Dependencies installed")

In [ ]:
# Verify data is present
import os
import json

train_file = 'datasets/gsm8k_split/llm_train.jsonl'
val_file = 'datasets/gsm8k_split/validation.jsonl'

if os.path.exists(train_file):
    with open(train_file) as f:
        count = sum(1 for _ in f)
    print(f"✓ Training data: {count} examples")
else:
    print("⚠ Training data not found. Upload datasets/gsm8k_split/ folder.")

if os.path.exists(val_file):
    with open(val_file) as f:
        count = sum(1 for _ in f)
    print(f"✓ Validation data: {count} examples")
else:
    print("⚠ Validation data not found.")

In [ ]:
# Prepare training data in CODI format
%cd codi_bundle

# Create datasets folder and copy data
!mkdir -p datasets/gsm8k
!cp ../datasets/gsm8k_split/llm_train.jsonl datasets/gsm8k/train.jsonl
!cp ../datasets/gsm8k_split/validation.jsonl datasets/gsm8k/val.jsonl
print("✓ Data prepared")

In [ ]:
# Start training (this will take ~2 hours)
# Adjust batch_size based on GPU memory: T4 = 4, A100 = 8+
!python train.py \
  --model_name_or_path gpt2 \
  --seed 42 \
  --model_max_length 512 \
  --lora_r 128 \
  --lora_alpha 32 \
  --lora_init \
  --num_latent 6 \
  --use_prj True \
  --prj_dim 768 \
  --inf_latent_iterations 6 \
  --remove_eos True \
  --use_lora True \
  --batch_size 4 \
  --data_name icot \
  --output_dir ../outputs/codi_trained \
  --num_train_epochs 3 \
  --learning_rate 0.0002

In [ ]:
# Monitor training (run this cell repeatedly to check progress)
!tail -n 30 ../outputs/codi_trained/*/logs.txt 2>/dev/null || echo "Waiting for logs..."

In [ ]:
# Check for errors
!find ../outputs/codi_trained -name "*.txt" -exec tail -n 20 {} \; | grep -i "error\|traceback" || echo "No errors found"

In [ ]:
# After training completes, check what was saved
%cd ..
!ls -lh outputs/codi_trained/

In [ ]:
# Download checkpoint to your local machine
!zip -r codi_checkpoint.zip outputs/codi_trained/
from google.colab import files
files.download('codi_checkpoint.zip')
print("✓ Download started - check your browser downloads")

## Next Steps

After downloading the checkpoint:

1. **Extract locally**:
   ```powershell
   Expand-Archive codi_checkpoint.zip -DestinationPath G:\Thesis\TokenSkip\
   ```

2. **Proceed to Phase 2** (Truth Vector Extraction):
   ```powershell
   python extract_truth_vector.py --steer-data datasets/gsm8k_split/steer_train.jsonl
   ```

3. **Phase 3** (Steering Inference):
   ```powershell
   python steer_inference.py --eval-data datasets/gsm8k_split/test.jsonl
   ```